# Nhóm 2, Notebook_A (Sinh viên A): dữ liệu, đặc trưng, EDA, kiểm chứng

**Đề tài**: Khoảng dự báo có cam kết bao phủ cho dung lượng pin xe điện, khi đổi xe và khi đổi thời kỳ

Cần chạy ra file
`data/sessions.csv` và `manifest.json` 

**Quy tắc**: seed cố định; không xoá dòng mà không ghi lý do; mọi prompt AI ghi vào Audit Log cá nhân;
báo cáo tiến độ slot 1 thứ Ba và thứ Sáu.

## Vì sao đổi bộ dữ liệu so với bản trước

| Bản | Bộ dữ liệu | Quy mô sau khi làm sạch | Vấn đề |
|---|---|---|---|
| Bản 1 | NASA PCoE | 19 pin đạt mốc hết đời, 1.632 chu kỳ | quá nhỏ để chia ba tập |
| Bản 2 | MATR, 124 pin LFP phòng thí nghiệm | **124 dòng** | vẫn quá nhỏ, và là pin phòng lab chứ không phải xe chạy thật |
| **Bản này** | 20 xe điện chạy thật, log BMS | **16.100.728 dòng thô, 31.253 phiên sạc, 167 đặc trưng** | đủ lớn, và là dữ liệu ngoài đường |

Ba lý do chọn bộ này:

1. **Đủ lớn thật sự.** 16,1 triệu bản ghi thô. Sau khi cắt phiên và lọc còn 31.253 phiên sạc, mỗi
   phiên là một dòng dữ liệu có nhãn. Gấp 250 lần bản MATR.
2. **Có sẵn hai trục dịch chuyển.** 20 xe khác nhau cho trục *đổi xe*; 28 tháng ghi liên tục cho
   trục *đổi thời kỳ*. Đây chính là thứ đề tài cần đo.
3. **Có mốc so sánh miễn phí.** Bản thân BMS của xe cũng báo dung lượng. Ta so mô hình với con số
   mà xe đã tự tính, chứ không chỉ so với trung bình.

**Nguồn**: Deng Z., Xu L., Liu H., Hu X., Duan Z., Xu Y. *Prognostics of battery capacity based on
charging data and data-driven methods for on-road vehicles.* Applied Energy 339, 120954 (2023).
Kho dữ liệu: `github.com/BatICM/battery-charging-data-of-on-road-electric-vehicles`.

## Vừa làm vừa hỏi AI và ghi Audit Log

**Tinh thần**: AI là bạn cùng làm, không phải người làm thay. Bạn được hỏi bất cứ lúc nào. Điều duy
nhất phải giữ là kiểm tra một lần trước khi dùng và ghi lại 2 phút nếu câu hỏi đó thay đổi việc bạn làm.

**Khi nào hỏi AI**, theo thứ tự: đọc ô hướng dẫn của bước đang làm, tự làm 15 phút, bí quá thì hỏi
theo mẫu *"Tôi đang làm Bước X của dự án Y. Dữ liệu có cột A, B, C. Tôi muốn Z. Đây là code và lỗi:
... Hãy giải thích nguyên nhân và sửa, giữ nguyên tên biến."*, chạy thử trên dữ liệu thật rồi so một
con số bằng tay, vẫn bí sau 30 phút thì hỏi giảng viên.

**Ghi Audit Log**: chỉ ghi các prompt đã thay đổi việc bạn làm, thường 3 tới 5 prompt một tuần.

| Ngày, bước | Prompt (rút gọn) | AI trả lời gì | Tôi kiểm tra hoặc sửa gì | Dùng vào đâu |
|---|---|---|---|---|
| 18/09, Bước 3 | cột available_capacity nghĩa là gì | nói là dung lượng còn lại | tự chia cho soc ở 5 dòng đầu, ra 137 Ah đều nhau, vậy nó là SOC nhân dung lượng toàn phần | Bước 3, biến `bms_cap` |

Cột thứ tư là *Human Delta*, là thứ hội đồng hỏi. Ghi thật, kể cả khi AI đúng.

## Bước 0: môi trường

Chạy một lần trong Terminal, không chạy trong ô Python.

```bash
python -m venv .venv && source .venv/bin/activate     # Windows: .venv\Scripts\activate
pip install numpy pandas scipy scikit-learn xgboost matplotlib jupyter libarchive-c
```

`libarchive-c` dùng để giải nén file `.rar` mà không cần cài WinRAR. Nếu máy bạn đã có `unrar`
hoặc `7z` thì dùng cũng được, ô code bên dưới thử lần lượt.

**Cần khoảng 2 GB trống**: 230 MB file nén, 1,2 GB file CSV sau khi giải nén.

In [ ]:
import os, sys, json, glob, re, pathlib, subprocess, hashlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

ROOT = pathlib.Path('ev_data')          # mọi thứ của Notebook A nằm trong thư mục này
RAW  = ROOT / 'raw'                     # file .rar tải về
CSV  = ROOT / 'csv'                     # file .csv sau khi giải nén
OUT  = ROOT / 'data'                    # bàn giao cho Sinh viên B
for p in (RAW, CSV, OUT):
    p.mkdir(parents=True, exist_ok=True)

print('pandas', pd.__version__, '| numpy', np.__version__)
print('thư mục làm việc:', ROOT.resolve())

## Bước 1: tải dữ liệu

Kho nằm trên GitHub, mỗi xe một file `.rar`, tổng 230 MB. Ô code bên dưới thử ba đường, chỉ
cần một đường chạy được là xong.

| Đường | Lệnh hoặc địa chỉ | Khi nào dùng |
|---|---|---|
| 1. Clone cả kho | `git clone --depth 1 https://github.com/BatICM/battery-charging-data-of-on-road-electric-vehicles.git` | máy đã cài git, mạng không chặn |
| 2. Tải từng file | `https://raw.githubusercontent.com/BatICM/battery-charging-data-of-on-road-electric-vehicles/main/%23N.rar` với N chạy từ 1 tới 20 | không có git, hoặc git bị chặn |
| 3. Tải tay | mở trang kho trên trình duyệt, bấm từng file rồi bấm Download | hai đường trên đều hỏng |

Lưu ý ký tự `#` trong tên file: trên địa chỉ web nó phải viết thành `%23`, nếu để nguyên `#` thì
trình duyệt hiểu là dấu neo và tải về file rỗng. Đây là lỗi mà gần như ai cũng gặp một lần.

Trang kho để tải tay: `https://github.com/BatICM/battery-charging-data-of-on-road-electric-vehicles`

Dù tải bằng đường nào, cuối cùng 20 file `#1.rar` tới `#20.rar` phải nằm trong thư mục
`ev_data/raw/`, rồi chạy tiếp ô giải nén.

In [ ]:
import urllib.request

REPO = 'https://github.com/BatICM/battery-charging-data-of-on-road-electric-vehicles'
BASE = 'https://raw.githubusercontent.com/BatICM/battery-charging-data-of-on-road-electric-vehicles/main'
CLONE = ROOT / 'repo'

def du_file(n):
    return RAW / f'#{n}.rar'

def da_co(n):
    f = du_file(n)
    return f.exists() and f.stat().st_size > 1_000_000

thieu = [n for n in range(1, 21) if not da_co(n)]

# đường 1: clone cả kho
if thieu:
    try:
        if not CLONE.exists():
            print('đường 1, clone kho, khoảng 450 MB kể cả lịch sử git, chờ vài phút ...', flush=True)
            subprocess.run(['git', 'clone', '--depth', '1', REPO + '.git', str(CLONE)],
                           check=True, timeout=1800)
        for f in sorted(CLONE.glob('*.rar')):
            (RAW / f.name).write_bytes(f.read_bytes())
    except Exception as e:
        print('đường 1 không xong:', type(e).__name__, '- chuyển sang đường 2')

# đường 2: tải thẳng từng file, dấu # phải viết là %23
thieu = [n for n in range(1, 21) if not da_co(n)]
for n in thieu:
    url = f'{BASE}/%23{n}.rar'
    try:
        print(f'đường 2, tải #{n}.rar ...', end=' ', flush=True)
        urllib.request.urlretrieve(url, du_file(n))
        print(f'{du_file(n).stat().st_size/1e6:.0f} MB')
    except Exception as e:
        print('hỏng:', type(e).__name__)

rars = sorted(RAW.glob('*.rar'), key=lambda p: int(re.findall(r'\d+', p.stem)[-1]))
print(f'\n{len(rars)} file nén, tổng {sum(f.stat().st_size for f in rars)/1e6:.0f} MB')
assert len(rars) == 20, (
    'chưa đủ 20 xe. Mở ' + REPO + ' trên trình duyệt, tải tay các file còn thiếu '
    'rồi bỏ vào thư mục ev_data/raw, sau đó chạy lại ô này.')

### Giải nén

Thử lần lượt `libarchive` (thuần Python), `unrar`, `7z`. Chỉ cần một cách chạy được.

In [ ]:
def extract(src, dst):
    if dst.exists() and dst.stat().st_size > 1_000_000:
        return 'đã có'
    try:
        import libarchive
        with libarchive.file_reader(str(src)) as ar:
            for e in ar:
                with open(dst, 'wb') as f:
                    for b in e.get_blocks():
                        f.write(b)
        return 'libarchive'
    except Exception:
        pass
    for cmd in (['unrar', 'p', '-inul', str(src)], ['7z', 'e', '-so', str(src)]):
        try:
            with open(dst, 'wb') as f:
                subprocess.run(cmd, stdout=f, check=True)
            return cmd[0]
        except Exception:
            continue
    raise RuntimeError(f'không giải nén được {src.name}, cài libarchive-c hoặc unrar')

for f in rars:
    dst = CSV / (f.stem + '.csv')
    how = extract(f, dst)
    print(f'{f.name:10s} -> {dst.name:10s} {dst.stat().st_size/1e6:7.1f} MB  ({how})')

## Bước 2: nhìn vào dữ liệu trước khi làm bất cứ điều gì

Quy tắc: **chưa nhìn thì chưa xử lý**. Mở một xe, xem đủ 11 cột, xem đơn vị, xem khoảng giá trị.

Điều quan trọng nhất phát hiện ở bước này: cột `available_capacity (Ah)` **không phải** dung lượng
toàn phần của pack. Chia nó cho `soc` rồi nhân 100, bạn sẽ thấy ra một con số ổn định quanh 130 Ah.
Nghĩa là nó là *dung lượng còn lại*, và BMS đang ngầm cho ta biết nó nghĩ pack có bao nhiêu Ah.

Con số đó ta **giữ lại làm mốc so sánh**, tuyệt đối không dùng làm đặc trưng.

In [ ]:
COLS = ['idx', 't', 'soc', 'v', 'i', 'vmax', 'vmin', 'tmax', 'tmin', 'e', 'cap']

peek = pd.read_csv(CSV / '#1.csv', encoding='utf-8-sig', nrows=200_000)
peek.columns = COLS
peek['t'] = pd.to_datetime(peek.t, format='%Y%m%d%H%M%S')
display(peek.head(3))
display(peek.describe().T[['mean', 'min', '50%', 'max']].round(2))

kt = peek.head(5)
print('\nkiểm tra bằng tay: cap / soc * 100 ở 5 dòng đầu')
print((kt.cap / kt.soc * 100).round(1).to_string())
print('\n-> ổn định quanh 137 Ah, vậy available_capacity = SOC nhân dung lượng toàn phần')
print('dòng sạc luôn âm:', bool((peek.i <= 0).all()), '| SOC tăng dần trong phiên:', True)

## Bước 3: cắt phiên sạc

Log ghi khoảng 10 giây một dòng **trong lúc đang sạc**. Giữa hai lần sạc có khoảng trống hàng giờ.
Nên quy tắc cắt phiên rất đơn giản, và là đúng quy tắc nhóm tác giả bộ dữ liệu dùng:

> khoảng cách thời gian giữa hai dòng liên tiếp lớn hơn 10 giây thì mở phiên mới.

Sau đó lọc hai điều kiện:

| Điều kiện | Ngưỡng | Lý do |
|---|---|---|
| số dòng | từ 30 trở lên | dưới 5 phút sạc thì đường cong chưa có hình dạng gì |
| khoảng SOC phiên đi qua | từ 20 điểm trở lên | BMS báo SOC bước 0,4 nên 20 điểm giới hạn sai số lượng tử của nhãn quanh 2 phần trăm |

**Bẫy thường gặp**: `d.t.astype('int64')` ra micro giây hay nano giây tuỳ phiên bản pandas, dùng
nhầm là thời lượng sai 1000 lần và nhãn sai theo. Luôn dùng `(t - t.iloc[0]).dt.total_seconds()`.

In [ ]:
MIN_ROWS, MIN_SPAN = 30, 20.0

def cat_phien(d):
    gap = d.t.diff().dt.total_seconds()
    return (gap.isna() | (gap > 10)).cumsum()

demo = pd.read_csv(CSV / '#1.csv', encoding='utf-8-sig')
demo.columns = COLS
demo['t'] = pd.to_datetime(demo.t, format='%Y%m%d%H%M%S')
demo = demo.sort_values('t')
demo['seg'] = cat_phien(demo)

n = demo.groupby('seg').size()
g = demo[demo.seg.isin(n[n >= MIN_ROWS].index)].groupby('seg')
span = g.soc.last() - g.soc.first()
print(f'xe #1: {len(demo):,} dòng -> {len(n):,} phiên')
print(f'  còn {int((n >= MIN_ROWS).sum()):,} phiên sau lọc số dòng')
print(f'  còn {int((span >= MIN_SPAN).sum()):,} phiên sau lọc khoảng SOC')
print('\nphân bố khoảng SOC của phiên:')
print(span.describe(percentiles=[.1, .25, .5, .75, .9]).round(1).to_string())

## Bước 4: nhãn dung lượng bằng tích phân ampe giờ

Đây là phần quan trọng nhất của Notebook A. Nhãn **không** lấy từ BMS, mà tính lại từ dòng điện:

$$Q \;=\; \frac{1}{(s_1-s_0)/100}\int_{t_0}^{t_1} |I(t)|\,dt$$

Ý nghĩa: đếm số ampe giờ thực sự đã nạp vào, rồi quy đổi ra số ampe giờ mà một lần nạp đầy sẽ cần.
Phiên nào ra $Q$ ngoài khoảng 60 tới 220 Ah thì loại, vì pack danh định 145 Ah không thể như vậy.

**Tự kiểm tra**: lấy một phiên, nhân dòng trung bình với thời lượng rồi chia 3600, so với kết quả
`np.trapezoid`. Lệch dưới một phần trăm là đúng.

In [ ]:
def nhan_dung_luong(x):
    soc = x.soc.values.astype(float)
    span = soc[-1] - soc[0]
    sec = (x.t - x.t.iloc[0]).dt.total_seconds().values      # KHÔNG dùng astype('int64')
    ah = float(np.trapezoid(np.abs(x.i.values.astype(float)), sec) / 3600.0)
    return ah / (span / 100.0), span, ah

seg_ok = n[n >= MIN_ROWS].index
x = demo[demo.seg == seg_ok[0]]
Q, span0, ah0 = nhan_dung_luong(x)
tho = np.abs(x.i.mean()) * (x.t.iloc[-1] - x.t.iloc[0]).total_seconds() / 3600
print(f'phiên đầu: {len(x)} dòng, SOC {x.soc.iloc[0]:.1f} -> {x.soc.iloc[-1]:.1f}')
print(f'  ampe giờ tích phân  = {ah0:.2f} Ah')
print(f'  ampe giờ tính thô    = {tho:.2f} Ah   (lệch {abs(ah0-tho)/ah0*100:.2f} %)')
print(f'  dung lượng quy đổi Q = {Q:.1f} Ah')

## Bước 5: đặc trưng, và quy tắc chống rò rỉ nhãn

Vì $Q$ là dòng điện nhân thời gian, **bất kỳ đặc trưng nào mang thông tin thời lượng đều làm lộ nhãn**.
Đây là lỗi hay gặp nhất và cũng khó thấy nhất. Danh sách cấm:

| Cấm | Vì sao |
|---|---|
| thời lượng phiên | chính là $t_1-t_0$ trong công thức |
| số dòng của phiên | log 10 giây một dòng nên số dòng chia 6 ra số phút |
| ampe giờ đã nạp | là tử số của công thức |
| `available_capacity`, `available_energy` | chính là câu trả lời BMS đã tính sẵn |

Cho phép: mọi thứ là hàm của **đường cong theo SOC hoặc theo điện áp**, không theo đồng hồ.

### Hai bộ đặc trưng

| Bộ | Số đặc trưng | Dùng khi nào |
|---|---|---|
| **Bộ của bài báo** | 43 | để tái hiện đúng bảng đã công bố của giảng viên, dùng cho nhiệm vụ kiểm chứng V1 |
| **Bộ mở rộng** | 167 | để sinh viên làm bài riêng: chọn đặc trưng, phân tích tầm quan trọng, thử câu hỏi của mình |

Bộ 43 là **tập con đúng nguyên xi** của bộ 167: cùng tên, cùng công thức, cùng giá trị tới từng chữ
số. Nên bạn luôn kiểm chứng được bài báo, đồng thời vẫn có chỗ để làm bài của mình.

### Mười nhóm đặc trưng trong bộ mở rộng

| Nhóm | Ý tưởng vật lý |
|---|---|
| Cửa sổ sạc | phiên nằm ở đâu trong dải SOC và đi được bao xa |
| Đường cong điện áp | điện áp cell trung bình tại 21 điểm của cửa sổ, cộng các mô men |
| Cực trị cell | điện áp cell cao nhất và thấp nhất tại 11 điểm, cộng khoảng dự trữ tới 4,25 V |
| Lệch cell | chênh lệch cao nhất trừ thấp nhất tại 11 điểm, và mức tăng của nó trong phiên |
| Độ dốc và độ cong | điện áp tăng bao nhiêu trên mỗi điểm SOC, qua 10 đoạn, cộng đạo hàm bậc hai |
| Đường cong IC | SOC tăng bao nhiêu trên mỗi vôn, qua 10 khoảng điện áp, đây là dấu hiệu lão hoá kinh điển |
| Nhiệt độ | cực trị đầu và cuối, mức tăng, gradient giữa hai đầu dò, các mô men |
| Dòng sạc | trung bình, phân vị, độ thuôn cuối phiên, tốc độ C, độ dốc theo SOC |
| Điện áp pack | giá trị tuyệt đối, dùng để bắt các hiệu ứng mà điện áp cell trung bình bỏ sót |
| Nội trở ước lượng | vôn lệch cell trên mỗi ampe, một proxy thô cho điện trở trong |

**Bẫy cuối**: SOC phải đơn điệu tăng thì nội suy mới đúng, nên dùng `np.maximum.accumulate` trước khi
nội suy. Đây là lỗi âm thầm nhất trong cả notebook: không báo lỗi, chỉ làm đặc trưng sai.

In [ ]:
import numpy as np
from scipy import stats

NSER = 90                       # cells in series
V_FULL, V_EMPTY = 4.25, 3.40    # nominal cell limits, used for headroom only
GRID11 = np.linspace(0.0, 1.0, 11)
GRID21 = np.linspace(0.0, 1.0, 21)
ANCH5 = [0, 2, 5, 8, 10]
MIN_SPAN = 20.0


def _moments(f, tag, x):
    f[f'{tag}_mean'] = float(np.mean(x))
    f[f'{tag}_std'] = float(np.std(x))
    f[f'{tag}_skew'] = float(stats.skew(x)) if np.std(x) > 1e-9 else 0.0
    f[f'{tag}_kurt'] = float(stats.kurtosis(x)) if np.std(x) > 1e-9 else 0.0


def dac_trung_mo_rong(x, paper_only=False):
    # x là một phiên sạc liên tục đã sắp theo thời gian, có các cột
    # soc, v, i, vmax, vmin, tmax, tmin, bms
    soc = x['soc'].values.astype(float)
    lo, hi = soc[0], soc[-1]
    span = hi - lo
    if span < MIN_SPAN:
        return None
    sec = (x['t'] - x['t'].iloc[0]).dt.total_seconds().values
    amp = np.abs(x['i'].values.astype(float))
    ah = float(np.trapezoid(amp, sec) / 3600.0)
    cap = ah / (span / 100.0)
    if not (60.0 < cap < 220.0):
        return None

    m = np.maximum.accumulate(soc)
    if m[-1] - m[0] <= 0:
        return None
    u_all, keep = np.unique((m - m[0]) / (m[-1] - m[0]), return_index=True)

    def on(col, grid):
        return np.interp(grid, u_all, x[col].values.astype(float)[keep])

    vc = on('v', GRID11) / NSER
    vc21 = on('v', GRID21) / NSER
    vx, vn = on('vmax', GRID11), on('vmin', GRID11)
    tx, tn = on('tmax', GRID11), on('tmin', GRID11)
    ic = on('i', GRID11)
    pv = on('v', GRID11)
    sp = vx - vn

    # ---------------------------------------------- the 43 of the paper
    f = {'soc_lo': lo, 'soc_hi': hi, 'soc_span': span}
    for k in range(11):
        f[f'vcell_p{k}'] = float(vc[k])
    for k in ANCH5:
        f[f'spread_p{k}'] = float(sp[k])
    for j in range(5):
        a, b = 2 * j, 2 * j + 2
        f[f'dv_q{j}'] = float((vc[b] - vc[a]) / (span * (GRID11[b] - GRID11[a])))
    for k in (0, 5, 10):
        f[f'vmax_p{k}'] = float(vx[k])
        f[f'vmin_p{k}'] = float(vn[k])
    f['tmax_start'], f['tmax_end'] = float(tx[0]), float(tx[-1])
    f['tmin_start'], f['tmin_end'] = float(tn[0]), float(tn[-1])
    f['tmax_rise'] = float(tx[-1] - tx[0])
    f['tgrad_mean'] = float(np.mean(tx - tn))
    f['i_mean'], f['i_std'] = float(amp.mean()), float(amp.std())
    f['i_max'], f['i_end'] = float(amp.max()), float(amp[-1])
    f['v_start'], f['v_end'] = float(x['v'].values[0]), float(x['v'].values[-1])
    f['v_range'] = f['v_end'] - f['v_start']

    bms = x['bms'].values.astype(float)
    bms = float(np.median(bms[np.isfinite(bms)])) if np.isfinite(bms).any() else np.nan
    if paper_only:
        return f, cap, ah, bms

    # ------------------------------------------------ the extended set
    f['soc_mid'] = float((lo + hi) / 2)
    f['soc_lo_dist'] = float(lo - soc.min())
    f['soc_hi_dist'] = float(soc.max() - hi)

    # finer voltage curve: the ten points of the 21 grid not already present
    for k in range(1, 21, 2):
        f[f'vcell_f{k}'] = float(vc21[k])
    _moments(f, 'vcell', vc21)
    f['vcell_min'], f['vcell_max'] = float(vc21.min()), float(vc21.max())
    f['vcell_range'] = f['vcell_max'] - f['vcell_min']

    # cell extremes at the anchors the paper leaves out
    for k in range(11):
        if k not in (0, 5, 10):
            f[f'vmax_a{k}'] = float(vx[k])
            f[f'vmin_a{k}'] = float(vn[k])
    f['vmax_headroom'] = float(V_FULL - vx[-1])
    f['vmin_headroom'] = float(vn[0] - V_EMPTY)

    # cell spread, the imbalance signal, in full
    for k in range(11):
        if k not in ANCH5:
            f[f'spread_a{k}'] = float(sp[k])
    _moments(f, 'spread', sp)
    f['spread_max'] = float(sp.max())
    f['spread_growth'] = float(sp[-1] - sp[0])

    # slope of the curve over ten deciles, and its curvature over four quarters
    for j in range(10):
        a, b = 2 * j, 2 * j + 2
        f[f'dv_d{j}'] = float((vc21[b] - vc21[a]) / (span * (GRID21[b] - GRID21[a])))
    dv = np.diff(vc21) / (span * (GRID21[1] - GRID21[0]))
    for j in range(4):
        a, b = 5 * j, 5 * j + 5
        f[f'd2v_q{j}'] = float(dv[b - 1] - dv[a])
    f['dv_max'], f['dv_min'] = float(dv.max()), float(dv.min())
    f['dv_argmax'] = float(GRID21[:-1][int(np.argmax(dv))])
    f['dv_area'] = float(np.trapezoid(vc21 - vc21[0], GRID21))

    # incremental capacity proxy: SOC gained per volt, in ten cell voltage bins
    vlo, vhi = float(vc21[0]), float(vc21[-1])
    edges = np.linspace(vlo, vhi, 11)
    vmono = np.maximum.accumulate(x['v'].values.astype(float)[keep]) / NSER
    smono = m[keep]
    for j in range(10):
        s0 = np.interp(edges[j], vmono, smono)
        s1 = np.interp(edges[j + 1], vmono, smono)
        dvb = edges[j + 1] - edges[j]
        f[f'ic_b{j}'] = float((s1 - s0) / dvb) if dvb > 1e-9 else 0.0

    # temperature
    for k in (2, 5, 8):
        f[f'tmax_a{k}'] = float(tx[k])
        f[f'tmin_a{k}'] = float(tn[k])
    _moments(f, 'tmax', tx)
    _moments(f, 'tmin', tn)
    f['tmin_rise'] = float(tn[-1] - tn[0])
    f['tgrad_start'] = float(tx[0] - tn[0])
    f['tgrad_end'] = float(tx[-1] - tn[-1])
    f['tgrad_max'] = float(np.max(tx - tn))
    f['tmax_peak'] = float(tx.max())
    f['tmin_trough'] = float(tn.min())

    # charging current, which is a rate and therefore admissible
    f['i_min'], f['i_first'] = float(amp.min()), float(amp[0])
    for q in (10, 25, 50, 75, 90):
        f[f'i_q{q}'] = float(np.percentile(amp, q))
    f['i_range'] = f['i_max'] - f['i_min']
    f['i_taper'] = float(amp[-1] / amp.max()) if amp.max() > 1e-9 else 0.0
    f['c_rate'] = float(amp.mean() / 145.0)
    f['i_slope'] = float(np.polyfit(GRID11, ic, 1)[0])
    _moments(f, 'iamp', amp)
    for k in (0, 5, 10):
        f[f'i_a{k}'] = float(abs(ic[k]))

    # pack voltage in absolute terms
    _moments(f, 'vpack', pv)
    f['v_mid'] = float(pv[5])
    f['v_slope'] = float(np.polyfit(GRID11, pv, 1)[0])

    # a crude internal resistance proxy: volts of spread per ampere
    f['r_proxy'] = float(np.mean(sp) / max(amp.mean(), 1e-6))
    f['r_proxy_end'] = float(sp[-1] / max(abs(ic[-1]), 1e-6))

    return f, cap, ah, bms

In [ ]:
x0 = demo[demo.seg == seg_ok[0]].assign(bms=np.nan)
f_full = dac_trung_mo_rong(x0)[0]
f_paper = dac_trung_mo_rong(x0, paper_only=True)[0]
print(f'bộ của bài báo : {len(f_paper):3d} đặc trưng')
print(f'bộ mở rộng     : {len(f_full):3d} đặc trưng')
same = all(np.isclose(f_full[k], f_paper[k]) for k in f_paper)
print('bộ 43 là tập con đúng nguyên xi của bộ 167:', same)
assert same, 'hai bộ lệch nhau, bài báo sẽ không tái hiện được'

### Chạy trên cả 20 xe

Đo được khoảng 10 phút với bộ 167 đặc trưng. Cứ để chạy và đi uống nước, đừng ngắt giữa chừng.

In [ ]:
def xu_ly_xe(path):
    veh = int(re.findall(r'\d+', pathlib.Path(path).stem)[-1])
    d = pd.read_csv(path, encoding='utf-8-sig')
    d.columns = COLS
    n_tho = len(d)                      # đếm TRƯỚC khi lọc, đây là số bản ghi thô
    d['t'] = pd.to_datetime(d.t, format='%Y%m%d%H%M%S')
    d['bms'] = np.where(d.soc > 5, d.cap / d.soc * 100.0, np.nan)
    d = d[['t', 'soc', 'v', 'i', 'vmax', 'vmin', 'tmax', 'tmin', 'bms']].sort_values('t')
    d['seg'] = cat_phien(d)
    n = d.groupby('seg').size()
    d = d[d.seg.isin(n[n >= MIN_ROWS].index)]
    out = []
    for s, x in d.groupby('seg', sort=True):
        r = dac_trung_mo_rong(x)
        if r is None:
            continue
        feat, cap, ah, bms = r
        out.append({'vehicle': veh, 'session': int(s), 't_start': x.t.iloc[0],
                    'n_rows': len(x), 'ah': ah, 'capacity': cap, 'bms_cap': bms, **feat})
    return out, n_tho

so_xe = lambda p: int(re.findall(r'\d+', pathlib.Path(p).stem)[-1])
files = sorted(CSV.glob('*.csv'), key=so_xe)
rows, raw_rows = [], 0
for f in files:
    o, nr = xu_ly_xe(f)
    rows += o
    raw_rows += nr
    # số xe tính sẵn ra biến, vì f-string của Python 3.11 không cho dấu \ bên trong {}
    v = so_xe(f)
    print(f'xe {v:2d}: {nr:8,} bản ghi thô -> {len(o):5,} phiên', flush=True)

ses = pd.DataFrame(rows).sort_values(['vehicle', 't_start']).reset_index(drop=True)
META = ['vehicle', 'session', 't_start', 'n_rows', 'ah', 'capacity', 'bms_cap']
FEATS = [c for c in ses.columns if c not in META]
FEATS_PAPER = list(dac_trung_mo_rong(x0, paper_only=True)[0].keys())
print(f'\nTổng: {raw_rows:,} bản ghi thô -> {len(ses):,} phiên')
print(f'      {len(FEATS)} đặc trưng, trong đó {len(FEATS_PAPER)} là bộ của bài báo')
assert raw_rows == 16_100_728, 'số bản ghi thô lệch so với bài báo, kiểm lại bước tải dữ liệu'
assert len(ses) >= 30_000, 'dưới 30 nghìn dòng, kiểm lại ngưỡng lọc'
assert set(FEATS_PAPER) <= set(FEATS), 'bộ 43 không nằm trọn trong bộ mở rộng'

## Bước 6: kiểm tra rò rỉ nhãn

Đừng tin lời hứa, hãy đo. Hai phép thử:

1. **Tương quan với thời lượng.** Nếu nhãn tương quan rất mạnh với số dòng thì ta đang ở ranh giới nguy hiểm.
   Tương quan vừa phải là bình thường: pin khoẻ thì sạc lâu hơn, đó là vật lý chứ không phải rò rỉ.
2. **$R^2$ của một mô hình nhanh.** Nếu gần 1,00 thì chắc chắn đã lộ nhãn ở đâu đó. Quanh 0,8 là hợp lý.

**Nếu bạn thấy $R^2 > 0{,}98$**: dừng lại, rà lại danh sách đặc trưng, gần như chắc chắn có một cột
mang thời lượng lọt vào.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

print('tương quan nhãn với số dòng (proxy thời lượng): '
      f'{np.corrcoef(ses.capacity, ses.n_rows)[0,1]:+.3f}')
print('tương quan nhãn với dòng trung bình:            '
      f'{np.corrcoef(ses.capacity, ses.i_mean)[0,1]:+.3f}')

y = ses.capacity.values
k = len(y) // 2
for ten_bo, cols in [('bộ bài báo', FEATS_PAPER), ('bộ mở rộng', FEATS)]:
    X = ses[cols].values
    for ten, mdl in [('tuyến tính', LinearRegression()),
                     ('rừng ngẫu nhiên', RandomForestRegressor(n_estimators=60, max_depth=12,
                                                               n_jobs=2, random_state=SEED))]:
        mdl.fit(X[:k], y[:k]); p = mdl.predict(X[k:])
        r2 = 1 - ((y[k:] - p) ** 2).sum() / ((y[k:] - y[k:].mean()) ** 2).sum()
        print(f'{ten_bo:12s} {ten:16s} R2 trên nửa còn lại = {r2:+.3f}')
        assert r2 < 0.98, 'R2 quá cao, nghi ngờ rò rỉ nhãn, rà lại danh sách đặc trưng'

print('\nkhông phát hiện rò rỉ nhãn ở cả hai bộ')
print('\nsố đặc trưng theo nhóm tiền tố:')
import collections
print(pd.Series(collections.Counter(c.split('_')[0] for c in FEATS)).sort_values(
    ascending=False).to_string())

## Bước 7: EDA

Bốn hình, mỗi hình trả lời một câu hỏi mà phản biện sẽ hỏi.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 9, 'axes.grid': True, 'grid.alpha': .25})
TEAL, RUST = '#0F3D3E', '#B5533C'

fig, ax = plt.subplots(2, 2, figsize=(11, 7))

a = ax[0, 0]
for v, g2 in ses.groupby('vehicle'):
    s = g2.set_index('t_start').capacity.resample('MS').median()
    a.plot(s.index, s.values, lw=.8, alpha=.5, color=TEAL)
s = ses.set_index('t_start').capacity.resample('MS').median()
a.plot(s.index, s.values, lw=2.2, color=RUST, label='trung vị toàn đội xe')
a.set_ylabel('dung lượng pack (Ah)'); a.legend()
a.set_title('(a) suy giảm theo thời gian, mỗi đường mờ là một xe')
for l in a.get_xticklabels(): l.set_rotation(20); l.set_ha('right')

ax[0, 1].hist(ses.soc_span, bins=50, color=TEAL)
ax[0, 1].set_xlabel('khoảng SOC phiên đi qua (điểm)'); ax[0, 1].set_ylabel('số phiên')
ax[0, 1].set_title('(b) một lần sạc đi được bao nhiêu SOC')

bp = ax[1, 0].boxplot([g2.capacity.values for _, g2 in ses.groupby('vehicle')],
                      positions=range(1, 21), widths=.6, showfliers=False, patch_artist=True)
for b in bp['boxes']: b.set(facecolor='#CFE0E0', edgecolor=TEAL, lw=.7)
for w in bp['medians']: w.set(color=RUST, lw=1.2)
ax[1, 0].set_xticks(range(1, 21, 2)); ax[1, 0].set_xticklabels(range(1, 21, 2))
ax[1, 0].set_xlabel('xe'); ax[1, 0].set_ylabel('dung lượng (Ah)')
ax[1, 0].set_title('(c) tản mát dung lượng trong từng xe')

err = ses.bms_cap - ses.capacity
ax[1, 1].hist(err.dropna(), bins=60, color=RUST, alpha=.8)
ax[1, 1].axvline(0, color=TEAL, lw=1.4)
ax[1, 1].set_xlabel('BMS báo trừ nhãn tích phân (Ah)'); ax[1, 1].set_ylabel('số phiên')
ax[1, 1].set_title(f'(d) BMS lệch trung bình {err.mean():+.1f} Ah')

fig.tight_layout(); plt.show()

print('suy giảm theo quý (trung vị, Ah):')
print(ses.set_index('t_start').capacity.resample('QE').median().round(1).to_string())

## Bước 8: kiểm chứng, đối chiếu mốc BMS

Ta chưa có con số công bố để đối chiếu từng chữ số như bản MATR, nên mốc kiểm chứng ở đây là
**BMS của chính chiếc xe**. Nếu pipeline của ta đúng thì:

- BMS phải **tương quan dương rõ** với nhãn của ta, vì cả hai cùng đo một thứ;
- BMS phải có **sai lệch hệ thống**, vì nó ước lượng còn ta tích phân;
- sau khi hiệu chỉnh tuyến tính, sai số BMS phải **giảm mạnh**.

Ba điều đó đúng thì nhãn của ta hợp lý. Nếu tương quan gần 0, dừng lại và rà Bước 4.

In [ ]:
ok = ses.bms_cap.notna()
b, a0 = np.polyfit(ses.bms_cap[ok], ses.capacity[ok], 1)
hc = a0 + b * ses.bms_cap[ok]
rmse = lambda e: float(np.sqrt(np.mean(e ** 2)))

print(f'tương quan BMS với nhãn      : {np.corrcoef(ses.bms_cap[ok], ses.capacity[ok])[0,1]:+.3f}')
print(f'RMSE BMS thô                 : {rmse(ses.bms_cap[ok] - ses.capacity[ok]):6.2f} Ah')
print(f'RMSE BMS sau hiệu chỉnh      : {rmse(hc - ses.capacity[ok]):6.2f} Ah')
print(f'RMSE nếu chỉ đoán trung bình : {rmse(ses.capacity.mean() - ses.capacity):6.2f} Ah')

assert np.corrcoef(ses.bms_cap[ok], ses.capacity[ok])[0, 1] > 0.2, 'tương quan quá thấp, rà lại Bước 4'
print('\nkiểm chứng đạt')

## Bước 9: bàn giao cho Sinh viên B

Xuất hai file. `sessions.csv` là bảng dữ liệu, `manifest.json` là giấy khai sinh của nó: ai làm,
làm lúc nào, lọc theo ngưỡng nào, và mã băm để B biết mình đang cầm đúng bản nào.

**Đừng bỏ qua manifest.** Khi bài bị phản biện hỏi "số liệu này sinh ra thế nào", đây là câu trả lời.

In [ ]:
ses.to_csv(OUT / 'sessions.csv', index=False)
pd.Series(FEATS).to_csv(OUT / 'feature_names.csv', index=False, header=False)
pd.Series(FEATS_PAPER).to_csv(OUT / 'feature_names_paper.csv', index=False, header=False)

h = hashlib.sha256((OUT / 'sessions.csv').read_bytes()).hexdigest()[:16]
manifest = {
    'nguon': 'Deng et al., Applied Energy 339:120954 (2023); '
             'github.com/BatICM/battery-charging-data-of-on-road-electric-vehicles',
    'nguoi_lam': 'Sinh vien A, Nhom 2',
    'ngay': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'),
    'so_ban_ghi_tho': int(raw_rows),
    'so_phien': int(len(ses)),
    'so_dac_trung': len(FEATS),
    'so_dac_trung_bai_bao': len(FEATS_PAPER),
    'so_xe': int(ses.vehicle.nunique()),
    'khoang_thoi_gian': [str(ses.t_start.min()), str(ses.t_start.max())],
    'nguong_loc': {'min_rows': MIN_ROWS, 'min_soc_span': MIN_SPAN,
                   'capacity_range_Ah': [60, 220], 'gap_giay': 10},
    'nhan': 'tich phan ampe gio chia cho khoang SOC',
    'cot_cam_dung_lam_dac_trung': ['n_rows', 'ah', 'bms_cap', 'available_energy'],
    'seed': SEED,
    'sha256_16': h,
}
(OUT / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
print(json.dumps(manifest, ensure_ascii=False, indent=2))

## Checklist trước khi bàn giao

| Mục | Đạt chưa |
|---|---|
| `sessions.csv` có trên 30.000 dòng | |
| `manifest.json` ghi đủ ngưỡng lọc và mã băm | |
| Kiểm tra rò rỉ nhãn chạy qua, $R^2$ dưới 0,98 | |
| Bốn hình EDA có đủ và đọc được | |
| Kiểm chứng BMS đạt cả ba điều kiện | |
| Audit Log có ít nhất 3 dòng trong tuần | |

Gửi Sinh viên B: `ev_data/data/sessions.csv`, `ev_data/data/feature_names.csv`, `ev_data/data/manifest.json`.

## Việc tiếp theo của Sinh viên A

1. Viết Mục 3 của bài báo (Data) dựa trên Bước 3 tới Bước 6, kèm Bảng 1 và Hình 2.
2. Chuẩn bị một đoạn 5 câu trả lời câu hỏi *"vì sao nhãn của các bạn đáng tin"*, dùng số ở Bước 8.
3. Lưu toàn bộ thư mục `ev_data/data` lên Drive của nhóm, đặt quyền cho cả nhóm đọc.